# ДЗ 2. Feature store для Customer Churn Prediction (за мотивами IBM Telco)

**Сценарій.** Телеком-оператор прогнозує, чи піде клієнт (churn) протягом 30 днів після моменту прогнозу `prediction_ts`.

**Доменний референс.** Набір *IBM Telco Customer Churn* використано лише як джерело понять і атрибутів (`gender`, `SeniorCitizen`, `Partner`, `Dependents`, послуги, `tenure`, `MonthlyCharges`, `TotalCharges`, `Contract`, `PaperlessBilling`, `Churn`).

> **Важливе обмеження.** Telco є snapshot-набором: у ньому немає точного часу відтоку. Тому в цьому ДЗ **temporal-історія ознак, `prediction_ts`, `label_horizon_end` і значення `churn_in_horizon` створені вручну як невеликий контрольований набір**. Припущення про реальні дати відтоку не робляться. `customer_id` — числовий сурогатний ключ замість рядкового `customerID` з Telco.

**Сутності домену:** `customers`, `services`, `customer_services` (bridge N:M).
**Offline:** `customer_features_offline` (історія ознак) і `training_events` (мітки).
**Online:** `customer_features_online` (один денормалізований рядок на клієнта).

Усі таблиці мають префікс `hw2_`.

## 0. Налаштування середовища (PostgreSQL 16 через `pgserver`)

In [ ]:
!pip install -q pgserver psycopg2-binary pandas

: 

In [ ]:
import pgserver, psycopg2, pandas as pd
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 30)

srv = pgserver.get_server("/tmp/hw2_pgdata")
conn = psycopg2.connect(srv.get_uri())
conn.autocommit = True

def q(sql):
    """Виконує SQL. SELECT повертає DataFrame, інші команди друкують статус."""
    with conn.cursor() as cur:
        cur.execute(sql)
        if cur.description:
            return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
        print("OK:", cur.statusmessage)

print(q("SELECT version()").iloc[0, 0])

## Завдання 1. ER-діаграма

 Нижче модель у вигляді схеми.

```
══════════════════════════ БІЗНЕСОВІ СУТНОСТІ ══════════════════════════════════
┌─────────────────────────┐ 1       N ┌──────────────────────────┐ N       1 ┌─────────────────────┐
│ hw2_customers           │───────────│ hw2_customer_services    │───────────│ hw2_services        │
│ PK customer_id          │ обов'язк. │ PK,FK customer_id        │ обов'язк. │ PK service_id       │
│    gender               │           │ PK,FK service_id         │           │    service_name     │
│    senior_citizen       │           │ PK    activated_at       │           │    service_group    │
│    has_partner          │           │       deactivated_at     │           └─────────────────────┘
│    has_dependents       │           └──────────────────────────┘
└───────────┬─────────────┘            (bridge для N:M customers <-> services)
            │
════════════╪════════════════════════ OFFLINE-РІВЕНЬ ═════════════════════════════
            │ 1 ── 0..N
            ├────────► ┌─────────────────────────────────────┐
            │          │ hw2_customer_features_offline       │
            │          │ PK,FK customer_id                   │
            │          │ PK    valid_from  (TIMESTAMPTZ)     │
            │          │       valid_to    (NULL = поточна)  │
            │          │       tenure_months, monthly_charges│
            │          │       total_charges (nullable)      │
            │          │       contract_type, paperless_billing│
            │          │       num_active_services           │
            │          └─────────────────────────────────────┘
            │ 1 ── 0..N
            ├────────► ┌─────────────────────────────────────┐
            │          │ hw2_training_events                 │
            │          │ PK event_id                         │
            │          │ FK customer_id                      │
            │          │    prediction_ts, label_horizon_end │
            │          │    churn_in_horizon  (target)       │
            │          └─────────────────────────────────────┘
════════════╪════════════════════════ ONLINE-РІВЕНЬ ═════════════════════════════
            │ 1 ── 0..1
            └────────► ┌─────────────────────────────────────┐
                       │ hw2_customer_features_online        │
                       │ PK,FK customer_id                   │
                       │    tenure_months, monthly_charges,  │
                       │    total_charges, contract_type,    │
                       │    paperless_billing,               │
                       │    num_active_services,             │
                       │    feature_ts, updated_at           │
                       └─────────────────────────────────────┘
```

**Кардинальності та обов'язковість.**
- `customers` 1 : N `customer_services` N : 1 `services` — **природний N:M**: клієнт підключає багато послуг, а кожну послугу мають багато клієнтів. Реалізовано bridge-таблицею `hw2_customer_services` з атрибутами зв'язку `activated_at` / `deactivated_at`. Рядок bridge без клієнта чи послуги неможливий (обидва FK обов'язкові); клієнт може не мати жодної послуги (0..N).
- `customers` 1 : 0..N `features_offline` — кожна версія ознак належить рівно одному клієнту.
- `customers` 1 : 0..N `training_events`.
- `customers` 1 : 0..1 `features_online` — не більше одного поточного рядка.

## Завдання 2. DDL (PostgreSQL 16)

In [ ]:
q("""
SET TIME ZONE 'UTC';

DROP TABLE IF EXISTS
    hw2_customer_features_online,
    hw2_training_events,
    hw2_customer_features_offline,
    hw2_customer_services,
    hw2_services,
    hw2_customers
CASCADE;
""")

In [ ]:
q("""
-- ===== Бізнесові сутності =====
CREATE TABLE hw2_customers (
    customer_id    BIGINT PRIMARY KEY,
    gender         VARCHAR(10) NOT NULL
        CHECK (gender IN ('Female', 'Male')),
    senior_citizen BOOLEAN NOT NULL,
    has_partner    BOOLEAN NOT NULL,
    has_dependents BOOLEAN NOT NULL
);

CREATE TABLE hw2_services (
    service_id    INTEGER PRIMARY KEY,
    service_name  VARCHAR(50) NOT NULL UNIQUE,
    service_group VARCHAR(20) NOT NULL
        CHECK (service_group IN ('phone', 'internet', 'addon'))
);

-- bridge-таблиця для N:M customers <-> services
CREATE TABLE hw2_customer_services (
    customer_id     BIGINT  NOT NULL REFERENCES hw2_customers(customer_id),
    service_id      INTEGER NOT NULL REFERENCES hw2_services(service_id),
    activated_at    DATE    NOT NULL,
    deactivated_at  DATE,
    PRIMARY KEY (customer_id, service_id, activated_at),
    CHECK (deactivated_at IS NULL OR deactivated_at > activated_at)
);

-- ===== OFFLINE-рівень =====
CREATE TABLE hw2_customer_features_offline (
    customer_id          BIGINT NOT NULL REFERENCES hw2_customers(customer_id),
    tenure_months        INTEGER NOT NULL CHECK (tenure_months >= 0),
    monthly_charges      NUMERIC(8, 2) NOT NULL CHECK (monthly_charges >= 0),
    total_charges        NUMERIC(10, 2)
        CHECK (total_charges IS NULL OR total_charges >= 0),   -- у Telco порожнє при tenure = 0
    contract_type        VARCHAR(20) NOT NULL
        CHECK (contract_type IN ('Month-to-month', 'One year', 'Two year')),
    paperless_billing    BOOLEAN NOT NULL,
    num_active_services  INTEGER NOT NULL CHECK (num_active_services >= 0),
    valid_from           TIMESTAMPTZ NOT NULL,
    valid_to             TIMESTAMPTZ,
    PRIMARY KEY (customer_id, valid_from),
    CHECK (valid_to IS NULL OR valid_to > valid_from)
);

-- основний access pattern: AS-OF пошук версії за (customer_id, момент часу)
CREATE INDEX hw2_idx_customer_features_asof
    ON hw2_customer_features_offline (customer_id, valid_from DESC);

CREATE TABLE hw2_training_events (
    event_id          BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    customer_id       BIGINT NOT NULL REFERENCES hw2_customers(customer_id),
    prediction_ts     TIMESTAMPTZ NOT NULL,
    label_horizon_end TIMESTAMPTZ NOT NULL,
    churn_in_horizon  BOOLEAN NOT NULL,
    CHECK (label_horizon_end > prediction_ts)
);

CREATE INDEX hw2_idx_training_events_customer
    ON hw2_training_events (customer_id, prediction_ts);

-- ===== ONLINE-рівень (денормалізований, 1 рядок на клієнта) =====
CREATE TABLE hw2_customer_features_online (
    customer_id          BIGINT PRIMARY KEY REFERENCES hw2_customers(customer_id),
    tenure_months        INTEGER NOT NULL CHECK (tenure_months >= 0),
    monthly_charges      NUMERIC(8, 2) NOT NULL CHECK (monthly_charges >= 0),
    total_charges        NUMERIC(10, 2)
        CHECK (total_charges IS NULL OR total_charges >= 0),
    contract_type        VARCHAR(20) NOT NULL
        CHECK (contract_type IN ('Month-to-month', 'One year', 'Two year')),
    paperless_billing    BOOLEAN NOT NULL,
    num_active_services  INTEGER NOT NULL CHECK (num_active_services >= 0),
    feature_ts           TIMESTAMPTZ NOT NULL,
    updated_at           TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);
""")

In [ ]:
q("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public' AND table_name LIKE 'hw2\\_%'
ORDER BY table_name;
""")

### Тестові дані (контрольований набір)

Значення модельовані за мотивами Telco (типові `MonthlyCharges`, `Contract`, послуги), але **історія версій і мітки створені вручну**. Інтервали версій безперервні: `valid_to` попередньої версії дорівнює `valid_from` наступної (півінтервал `[valid_from, valid_to)` гарантує відсутність перекриття). Усі часові літерали явно в UTC (`+00`).

In [ ]:
q("""
INSERT INTO hw2_customers (customer_id, gender, senior_citizen, has_partner, has_dependents) VALUES
    (1, 'Female', FALSE, TRUE,  FALSE),
    (2, 'Male',   FALSE, FALSE, FALSE),
    (3, 'Male',   FALSE, FALSE, FALSE),
    (4, 'Female', TRUE,  TRUE,  TRUE);

INSERT INTO hw2_services (service_id, service_name, service_group) VALUES
    (1, 'PhoneService',    'phone'),
    (2, 'InternetService', 'internet'),
    (3, 'OnlineSecurity',  'addon'),
    (4, 'OnlineBackup',    'addon'),
    (5, 'DeviceProtection','addon'),
    (6, 'TechSupport',     'addon'),
    (7, 'StreamingTV',     'addon'),
    (8, 'StreamingMovies', 'addon');

INSERT INTO hw2_customer_services (customer_id, service_id, activated_at, deactivated_at) VALUES
    (1, 1, DATE '2024-09-01', NULL),
    (1, 2, DATE '2024-11-01', NULL),
    (2, 1, DATE '2021-12-01', NULL),
    (2, 2, DATE '2021-12-01', NULL),
    (2, 3, DATE '2022-03-01', NULL),
    (3, 1, DATE '2024-10-15', NULL),
    (3, 2, DATE '2024-10-15', NULL),
    (3, 7, DATE '2024-12-15', NULL),
    (4, 1, DATE '2023-12-01', NULL),
    (4, 2, DATE '2023-12-01', NULL),
    (4, 6, DATE '2024-02-01', DATE '2024-12-01'),
    (4, 3, DATE '2024-12-01', NULL);
""")

In [ ]:
q("""
INSERT INTO hw2_customer_features_offline
    (customer_id, tenure_months, monthly_charges, total_charges, contract_type,
     paperless_billing, num_active_services, valid_from, valid_to)
VALUES
    -- клієнт 1: 4 послідовні версії (Month-to-month)
    (1,  1, 29.85,   29.85, 'Month-to-month', TRUE, 1, TIMESTAMPTZ '2024-09-01 00:00:00+00', TIMESTAMPTZ '2024-10-01 00:00:00+00'),
    (1,  2, 29.85,   59.70, 'Month-to-month', TRUE, 1, TIMESTAMPTZ '2024-10-01 00:00:00+00', TIMESTAMPTZ '2024-11-01 00:00:00+00'),
    (1,  3, 44.85,  104.55, 'Month-to-month', TRUE, 2, TIMESTAMPTZ '2024-11-01 00:00:00+00', TIMESTAMPTZ '2024-12-01 00:00:00+00'),
    (1,  4, 44.85,  149.40, 'Month-to-month', TRUE, 2, TIMESTAMPTZ '2024-12-01 00:00:00+00', NULL),
    -- клієнт 2: 3 версії (зміна контракту One year -> Two year)
    (2, 34, 56.95, 1889.50, 'One year',  FALSE, 3, TIMESTAMPTZ '2024-06-01 00:00:00+00', TIMESTAMPTZ '2024-10-01 00:00:00+00'),
    (2, 38, 56.95, 2117.30, 'One year',  FALSE, 3, TIMESTAMPTZ '2024-10-01 00:00:00+00', TIMESTAMPTZ '2024-11-15 00:00:00+00'),
    (2, 39, 52.95, 2170.25, 'Two year',  FALSE, 3, TIMESTAMPTZ '2024-11-15 00:00:00+00', NULL),
    -- клієнт 3: 3 версії (total_charges = NULL при tenure = 0, як у Telco)
    (3,  0, 53.85,    NULL, 'Month-to-month', TRUE, 2, TIMESTAMPTZ '2024-10-15 00:00:00+00', TIMESTAMPTZ '2024-11-15 00:00:00+00'),
    (3,  1, 53.85,   53.85, 'Month-to-month', TRUE, 2, TIMESTAMPTZ '2024-11-15 00:00:00+00', TIMESTAMPTZ '2024-12-15 00:00:00+00'),
    (3,  2, 70.70,  124.55, 'Month-to-month', TRUE, 3, TIMESTAMPTZ '2024-12-15 00:00:00+00', NULL),
    -- клієнт 4: 2 версії
    (4, 12, 89.10, 1020.00, 'Two year',  FALSE, 3, TIMESTAMPTZ '2024-08-01 00:00:00+00', TIMESTAMPTZ '2024-12-01 00:00:00+00'),
    (4, 16, 95.40, 1401.60, 'Two year',  FALSE, 3, TIMESTAMPTZ '2024-12-01 00:00:00+00', NULL);

-- мітки створені вручну (у Telco немає часу відтоку)
INSERT INTO hw2_training_events
    (customer_id, prediction_ts, label_horizon_end, churn_in_horizon)
VALUES
    (1, TIMESTAMPTZ '2024-11-10 10:00:00+00', TIMESTAMPTZ '2024-12-10 10:00:00+00', FALSE),
    (1, TIMESTAMPTZ '2024-12-05 10:00:00+00', TIMESTAMPTZ '2025-01-04 10:00:00+00', TRUE),
    (2, TIMESTAMPTZ '2024-11-01 09:00:00+00', TIMESTAMPTZ '2024-12-01 09:00:00+00', FALSE),
    (2, TIMESTAMPTZ '2024-12-01 09:00:00+00', TIMESTAMPTZ '2024-12-31 09:00:00+00', FALSE),
    (3, TIMESTAMPTZ '2024-11-15 00:00:00+00', TIMESTAMPTZ '2024-12-15 00:00:00+00', FALSE),
    (3, TIMESTAMPTZ '2024-12-20 12:00:00+00', TIMESTAMPTZ '2025-01-19 12:00:00+00', TRUE),
    (4, TIMESTAMPTZ '2024-11-01 09:00:00+00', TIMESTAMPTZ '2024-12-01 09:00:00+00', FALSE),
    (4, TIMESTAMPTZ '2024-12-01 00:00:00+00', TIMESTAMPTZ '2024-12-31 00:00:00+00', FALSE);
""")

In [ ]:
q("""
SELECT * FROM hw2_customer_features_offline ORDER BY customer_id, valid_from;
""")

### Наповнення online-таблиці з offline-джерела

В online потрапляє лише **поточна** версія (`valid_to IS NULL`). Запит ідемпотентний (`UPSERT`), тому його можна запускати повторно після кожного оновлення offline-даних.

In [ ]:
q("""
INSERT INTO hw2_customer_features_online
    (customer_id, tenure_months, monthly_charges, total_charges, contract_type,
     paperless_billing, num_active_services, feature_ts)
SELECT customer_id, tenure_months, monthly_charges, total_charges, contract_type,
       paperless_billing, num_active_services, valid_from
FROM hw2_customer_features_offline
WHERE valid_to IS NULL
ON CONFLICT (customer_id) DO UPDATE SET
    tenure_months       = EXCLUDED.tenure_months,
    monthly_charges     = EXCLUDED.monthly_charges,
    total_charges       = EXCLUDED.total_charges,
    contract_type       = EXCLUDED.contract_type,
    paperless_billing   = EXCLUDED.paperless_billing,
    num_active_services = EXCLUDED.num_active_services,
    feature_ts          = EXCLUDED.feature_ts,
    updated_at          = CURRENT_TIMESTAMP;
""")

In [ ]:
q("SELECT * FROM hw2_customer_features_online ORDER BY customer_id;")

## Завдання 3. Нормалізація

**3НФ.** `hw2_customers`, `hw2_services`, `hw2_customer_services`, `hw2_customer_features_offline` і `hw2_training_events` перебувають у 3НФ: значення всіх атомарні, кожен неключовий атрибут залежить від усього ключа й не залежить від інших неключових атрибутів. Транзитивних залежностей немає: наприклад, група послуги винесена в `services` (`service_id → service_name, service_group`), а не повторюється в кожному рядку зв'язку.

**Функціональні залежності.** `customer_id → gender, senior_citizen, has_partner, has_dependents`. `service_id → service_name, service_group`. Зв'язок N:M розкладено через bridge: `(customer_id, service_id, activated_at) → deactivated_at`. Для історії ознак: `(customer_id, valid_from) → valid_to, tenure_months, monthly_charges, total_charges, contract_type, paperless_billing, num_active_services`. Для подій: `event_id → customer_id, prediction_ts, label_horizon_end, churn_in_horizon`.

**Денормалізація.** `hw2_customer_features_online` навмисно денормалізована: усі ознаки в одному рядку за `customer_id`, без історії. `num_active_services` у feature-таблицях є похідним значенням, яке можна порахувати з bridge; його зберігаємо готовим, щоб не робити JOIN-ів і агрегації під час читання. Це виправдано read pattern онлайн-інференсу: lookup за `entity_id` з низькою latency.

**Ризик.** Дублювання між offline, online і bridge може призвести до розбіжностей (stale features, training-serving skew). Свіжість видно за `feature_ts` та `updated_at`.

**Оновлення.** Online перезаповнюється UPSERT-запитом з поточних версій offline (`valid_to IS NULL`). Offline лишається єдиним джерелом істини.

## Завдання 4. Point-in-Time correctness

### AS-OF #1. Клієнт 1 на `2024-11-10 10:00:00+00` (очікується версія від 2024-11-01)

In [ ]:
q("""
SELECT customer_id, tenure_months, monthly_charges, total_charges, contract_type,
       num_active_services, valid_from, valid_to
FROM hw2_customer_features_offline
WHERE customer_id = 1
  AND valid_from <= TIMESTAMPTZ '2024-11-10 10:00:00+00'
  AND (valid_to > TIMESTAMPTZ '2024-11-10 10:00:00+00' OR valid_to IS NULL);
""")

### AS-OF #2. Клієнт 1 на `2024-12-05 10:00:00+00` (очікується версія від 2024-12-01)

In [ ]:
q("""
SELECT customer_id, tenure_months, monthly_charges, total_charges, contract_type,
       num_active_services, valid_from, valid_to
FROM hw2_customer_features_offline
WHERE customer_id = 1
  AND valid_from <= TIMESTAMPTZ '2024-12-05 10:00:00+00'
  AND (valid_to > TIMESTAMPTZ '2024-12-05 10:00:00+00' OR valid_to IS NULL);
""")

### AS-OF #3. Усі клієнти на межі версій: `2024-12-01 00:00:00+00`

Цей момент збігається з `valid_from` версій клієнтів 1 і 4 та лежить усередині інтервалів клієнтів 2 і 3. Для клієнтів 1 і 4 очікуються нові версії (`valid_from` входить в інтервал), а не попередні (`valid_to` не входить).

In [ ]:
q("""
SELECT customer_id, tenure_months, monthly_charges, total_charges, contract_type,
       num_active_services, valid_from, valid_to
FROM hw2_customer_features_offline
WHERE valid_from <= TIMESTAMPTZ '2024-12-01 00:00:00+00'
  AND (valid_to > TIMESTAMPTZ '2024-12-01 00:00:00+00' OR valid_to IS NULL)
ORDER BY customer_id;
""")

### AS-OF JOIN. Формування training data за `prediction_ts`

In [ ]:
training = q("""
SELECT
    e.event_id, e.customer_id, e.prediction_ts, e.label_horizon_end, e.churn_in_horizon,
    f.tenure_months, f.monthly_charges, f.total_charges, f.contract_type,
    f.paperless_billing, f.num_active_services,
    f.valid_from AS feature_valid_from, f.valid_to AS feature_valid_to
FROM hw2_training_events AS e
LEFT JOIN hw2_customer_features_offline AS f
    ON f.customer_id = e.customer_id
   AND f.valid_from <= e.prediction_ts
   AND (f.valid_to > e.prediction_ts OR f.valid_to IS NULL)
ORDER BY e.event_id;
""")
training

Перевірка: кожна подія має рівно один рядок результату, і версія ознак завжди починається не пізніше `prediction_ts`.

In [ ]:
n_events = q("SELECT COUNT(*) AS n FROM hw2_training_events").iloc[0, 0]
assert len(training) == n_events, "AS-OF JOIN створив дублікати або втратив події"
assert training["feature_valid_from"].notna().all(), "Для події не знайдено версію ознак"
assert (training["feature_valid_from"] <= training["prediction_ts"]).all(), "Витік майбутнього!"
print(f"OK: {len(training)} подій, дублікатів немає, майбутніх версій немає")

### Перевірка перекривних інтервалів (очікується 0 рядків)

In [ ]:
overlaps = q("""
SELECT a.customer_id,
       a.valid_from AS a_from, a.valid_to AS a_to,
       b.valid_from AS b_from, b.valid_to AS b_to
FROM hw2_customer_features_offline AS a
JOIN hw2_customer_features_offline AS b
  ON a.customer_id = b.customer_id
 AND a.valid_from < COALESCE(b.valid_to, 'infinity'::timestamptz)
 AND b.valid_from < COALESCE(a.valid_to, 'infinity'::timestamptz)
 AND a.valid_from < b.valid_from;
""")
print("Перекриттів:", len(overlaps))
assert len(overlaps) == 0
overlaps

### Додаткова перевірка обов'язкових вимог (кількість рядків, сутностей, версій)

In [ ]:
q("""
SELECT
    COUNT(*)                    AS history_rows,
    COUNT(DISTINCT customer_id) AS entities,
    MAX(versions)               AS max_versions_per_entity
FROM (
    SELECT customer_id, COUNT(*) OVER (PARTITION BY customer_id) AS versions
    FROM hw2_customer_features_offline
) t;
""")

### Демонстрація неправильного JOIN (лише за `customer_id`)

In [ ]:
q("""
SELECT e.event_id, e.customer_id, e.prediction_ts,
       f.valid_from AS feature_valid_from, f.tenure_months, f.total_charges,
       (f.valid_from > e.prediction_ts) AS from_the_future
FROM hw2_training_events AS e
JOIN hw2_customer_features_offline AS f ON f.customer_id = e.customer_id
WHERE e.event_id = 1
ORDER BY f.valid_from;
""")

In [ ]:
q("""
SELECT
    (SELECT COUNT(*) FROM hw2_training_events) AS events,
    COUNT(*)                                   AS rows_after_naive_join,
    COUNT(*) FILTER (WHERE f.valid_from > e.prediction_ts) AS rows_with_future_features
FROM hw2_training_events AS e
JOIN hw2_customer_features_offline AS f ON f.customer_id = e.customer_id;
""")

### Пояснення

**1. Чому `valid_from <= prediction_ts`.** Версія ознак могла бути використана для прогнозу лише якщо вона вже існувала на момент прогнозу. Версії, що починаються пізніше, відсікаються. Нестрога нерівність потрібна, бо момент `valid_from` входить в інтервал: якщо версія почалася рівно в `prediction_ts`, вона вже чинна (так для події клієнта 4 на `2024-12-01 00:00:00+00`).

**2. Чому `valid_to > prediction_ts OR valid_to IS NULL`.** Версія чинна, поки не замінена наступною. Строге `>` означає, що в момент `valid_to` стара версія вже недійсна. `valid_to IS NULL` позначає поточну, ще не закриту версію; без `OR IS NULL` вона б ніколи не потрапила в результат, бо порівняння з NULL дає UNKNOWN.

**3. Чому півінтервал `[valid_from, valid_to)`.** Кожен момент часу належить **рівно одній** версії. Якби обидві межі були включені, момент `valid_to` попередньої версії збігався б із `valid_from` наступної, і AS-OF JOIN повернув би два рядки (дублікат).

**4. Що б потрапило з майбутнього без часового фільтра.** Простий JOIN за `customer_id` приєднує до кожної події всі версії клієнта, зокрема пізніші. Для події 1 (клієнт 1, `2024-11-10`) до training-набору потрапила б версія від `2024-12-01` (`tenure_months = 4`, `total_charges = 149.40`), якої на момент прогнозу ще не існувало. Наслідки: (а) дублікати training examples, бо рядків більше, ніж подій; (б) data leakage, бо модель «бачить» стан клієнта після прогнозу, зокрема накопичені платежі, що корелюють із міткою; (в) завищена якість моделі на навчанні та валідації, яка не відтвориться в production.

> Нагадування: мітки й історія тут створені вручну, тому ці приклади демонструють механіку point-in-time, а не реальні закономірності відтоку.

## Завдання 5. Reflection

**Що було найскладнішим.** Найскладнішою виявилась реалізація point-in-time correctness. Було легко написати `BETWEEN valid_from AND valid_to`, але це дає дублікати на межі версій. Рішенням став півінтервал `[valid_from, valid_to)` та явна обробка `valid_to IS NULL`. Окремою проблемою було те, що Telco є snapshot-набором без часу відтоку: довелося чесно створити контрольовану історію й мітки та не видавати їх за реальні дані.

**Проєктні компроміси.** Бізнесові сутності та offline-історію я нормалізула до 3НФ: довідник послуг винесений в окрему таблицю, а N:M між клієнтами та послугами реалізований bridge-таблицею, тож кожен факт змінюється в одному місці. Online-таблицю навмисно денормалізував: один рядок на `customer_id` з готовими ознаками, зокрема `num_active_services`. Це дає читання одним lookup без JOIN, що відповідає інференсу з низькою latency. Ціна цього компромісу — дублювання й ризик розходження з offline.

**Швидший online-рівень.** Для великої кількості користувачів я б зберігала попередньо матеріалізований рядок ознак за `entity_id` у key-value сховищі з кешуванням гарячих ключів. Таблицю шардував би за `entity_id` для горизонтального масштабування, а оновлення виконував би пакетно або стрімінгом з offline-джерела. Свіжість контролював би через `feature_ts` з порогом допустимого віку.

**Ризики еволюції схеми за шість місяців.** Перейменування ознак ламає запити й моделі, тому потрібні версії feature-схеми. Зміна типів (наприклад, `total_charges` з `NUMERIC` на інший формат) вимагає міграції історії. Найнебезпечніша зміна семантики: `monthly_charges` з тією ж назвою, але інакше обчислений (із податками чи без), мовчки зіпсує порівнянність старих і нових значень. Видалення колонок ламає старі версії моделей, тому колонки слід позначати deprecated і зберігати на період міграції, а нові ознаки додавати окремими колонками.